In [0]:
%sql
CREATE OR REPLACE TEMP VIEW episode_candidate AS
SELECT
  co.person_id,
  MIN(co.condition_start_date) AS episode_start_date,
  CAST(MIN(co.condition_start_date) AS TIMESTAMP) AS episode_start_datetime,
  CAST(NULL AS DATE) AS episode_end_date,
  CAST(NULL AS TIMESTAMP) AS episode_end_datetime,
  CAST(NULL AS BIGINT) AS episode_parent_id,
  CAST(1 AS INT) AS episode_number,
  CAST(201826 AS INT) AS episode_object_concept_id, -- Type 2 diabetes mellitus
  CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'episode',
    't2dm_outpatient',
    CAST(co.person_id AS STRING),
    CAST(MIN(co.condition_start_date) AS STRING)
  ) AS episode_source_value
FROM _exponent.omop_tw.condition_occurrence co
WHERE co.condition_concept_id = 201826
GROUP BY co.person_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW episode_event_candidate AS

SELECT
  e.episode_id,
  co.condition_occurrence_id AS event_id,
  1147127 AS episode_event_field_concept_id
FROM _exponent.omop_tw.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_tw.condition_occurrence co
  ON co.person_id = ec.person_id
WHERE co.condition_start_date >= ec.episode_start_date
  AND co.condition_concept_id = 201826

UNION ALL

SELECT
  e.episode_id,
  vo.visit_occurrence_id AS event_id,
  1147126 AS episode_event_field_concept_id
FROM _exponent.omop_tw.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_tw.visit_occurrence vo
  ON vo.person_id = ec.person_id
WHERE vo.visit_start_date >= ec.episode_start_date

UNION ALL

SELECT
  e.episode_id,
  m.measurement_id AS event_id,
  1147130 AS episode_event_field_concept_id
FROM _exponent.omop_tw.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_tw.measurement m
  ON m.person_id = ec.person_id
WHERE m.measurement_date >= ec.episode_start_date

UNION ALL

SELECT
  e.episode_id,
  de.drug_exposure_id AS event_id,
  1147132 AS episode_event_field_concept_id
FROM _exponent.omop_tw.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_tw.drug_exposure de
  ON de.person_id = ec.person_id
WHERE de.drug_exposure_start_date >= ec.episode_start_date;

In [0]:
%sql
INSERT INTO _exponent.omop_tw.episode_event (
  episode_id,
  episode_event_field_concept_id
)
SELECT
  episode_id,
  episode_event_field_concept_id
FROM episode_event_candidate;